In [8]:
import os
import shutil
import gc
import json
import time
import copy
import toml
import torch
import joblib
import datetime
import argparse
import subprocess

import pandas as pd
import numpy as np

from config_io import Config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from GPUtil import showUtilization as gpu_usage
# from config import configs # run once to ensure the latest configs are loaded
# print("Finish loading configs")
# from config_small_scale import NETGPT_BASE_FOLDER
from torch.profiler import profile, record_function, ProfilerActivity
import netshare.ray as ray
from netshare import Generator

In [9]:
work_folder = "./netshare_work_folder"
if os.path.exists(work_folder):
    shutil.rmtree(work_folder)
os.makedirs(work_folder, exist_ok=True)

ray.config.enabled = False
ray.init(address="auto")
generator = Generator(config="netshare_config.json")
generator.train_and_generate(work_folder=work_folder)
print(f"Generated file list: {generator._pre_post_processor.best_syndf_filename_list}")
print(f"Best Gen trace: {generator._pre_post_processor.best_syndf_filename_list[0]}")
syn_df = pd.read_csv(generator._pre_post_processor.best_syndf_filename_list[0])
syn_df.to_csv("../data/generated_netshare_caida_10k.csv", index=False)
print("Saved generated data to ../data/generated_netshare_caida_10k.csv")
ray.shutdown()

Ray is disabled
NetsharePrePostProcessor._pre_process
../data/caida-10k.csv
dataset type: pcap
        srcip       dstip  srcport  dstport proto              time  pkt_len  \
0  3992027753  2325249714      443    33195   TCP  1521118750461923     1400   
1  3223949995  2332752498    50527      443   TCP  1521118750461924       40   
2  1056685515  3618636787      443    27308   TCP  1521118750461924       52   
3   284281003  1074109364       80    63403   TCP  1521118750461924     1500   
4   568626588  3261365128      443    50749   TCP  1521118750461928       44   

   version  ihl  tos     id  flag  off  ttl  chksum  
0        4    5    0  27106     2    0   88   14840  
1        4    5    0   8563     2    0  115    3595  
2        4    5    0   8751     2    0  234   38442  
3        4    5    8  45145     2    0   59   54372  
4        4    5    0      0     2    0   54   22367  
metadata cols: ['srcip', 'dstip', 'srcport', 'dstport', 'proto']
word2vec cols: ['srcport', 'dstport

11/21/2025 15:49:14:WARNING:consider setting layer size to a multiple of 4 for greater performance
11/21/2025 15:49:15:WARNING:under 10 jobs per worker: consider setting a smaller `batch_words' for smoother alpha decay


Word2Vec model is saved at ./netshare_work_folder/pre_processed_data/word2vec_vecSize_10.model
Building annoy dictionary word2vec...
{'port': ['srcport', 'dstport'], 'proto': ['proto']}
Finish building Angular trees...
metadata fields: ['srcip', 'dstip', 'srcport', 'dstport', 'proto']
timeseries fields: ['pkt_len', 'tos', 'id', 'flag', 'off', 'ttl']
Using fixed_time
1
Chunk_id: 0, # of pkts/records: 10000
df_chunk_cnt_validation: 10000
Chunk time: 0.02287 seconds
compute flowkey-chunk list from scratch...
processing chunk 1/1, # of flows: 3374
# of total flows: 3374
# of total flows (sanity check): 3374
# of flows cross chunk (of total flows): 0 (0.0%)
# of non-continuous flows: 0
chunk_id: 0, max_flow_len: 419
global max flow len: 419
Top 10 per-chunk flow length: [70, 132, 149, 156, 166, 184, 219, 303, 342, 419]


0it [00:00, ?it/s]


Chunk_id: 0
Before truncation, df_per_chunk: (10000, 15)


...

NetShareManager._train
Number of valid chunks: 1
Number of configurations after expanded: 1
[{'dp_noise_multiplier': None, 'dp': False, 'pretrain': True, 'config_ids': [0]}]
Config group 0: DP: False, pretrain: True
Start launching chunk0 experiments...
DoppelGANgerTorchModel._train
Currently training with config: {'overwrite': True, 'original_data_file': '../data/caida-10k.csv', 'dataset_type': 'pcap', 'n_chunks': 1, 'dp': False, 'allowed_data_types': ['ip_string', 'integer', 'float', 'string'], 'allowed_data_encodings': ['categorical', 'bit', 'word2vec_port', 'word2vec_proto'], 'pretrain_dir': './netshare_work_folder/models/chunkid-0/sample_len-10/checkpoint/epoch_id-19.pt', 'skip_chunk0_train': False, 'pretrain_non_dp': True, 'pretrain_non_dp_reduce_time': 4.0, 'pretrain_dp': False, 'run': 0, 'batch_size': 64, 'sample_len': 10, 'sample_len_expand': True, 'iteration': 200000, 'vis_freq': 100000, 'vis_num_sample': 5, 'd_rounds': 5, 'g_rounds': 1, 'num_packing': 1, 'noise': True, 'attr

100%|##########| 20/20 [02:55<00:00,  8.78s/it]


Finish launching chunk0 experiments ...
Number of valid chunks: 1
Number of configurations after expanded: 1
Start generating attributes ...
DoppelGANgerTorchModel._generate
Currently generating with config: {'overwrite': True, 'original_data_file': '../data/caida-10k.csv', 'dataset_type': 'pcap', 'n_chunks': 1, 'dp': False, 'allowed_data_types': ['ip_string', 'integer', 'float', 'string'], 'allowed_data_encodings': ['categorical', 'bit', 'word2vec_port', 'word2vec_proto'], 'pretrain_dir': './netshare_work_folder/models/chunkid-0/sample_len-10/checkpoint/epoch_id-19.pt', 'skip_chunk0_train': False, 'pretrain_non_dp': True, 'pretrain_non_dp_reduce_time': 4.0, 'pretrain_dp': False, 'run': 0, 'batch_size': 64, 'sample_len': 10, 'sample_len_expand': True, 'iteration': 200000, 'vis_freq': 100000, 'vis_num_sample': 5, 'd_rounds': 5, 'g_rounds': 1, 'num_packing': 1, 'noise': True, 'attr_noise_type': 'normal', 'feature_noise_type': 'normal', 'rnn_mlp_num_layers': 0, 'feed_back': False, 'g_lr':

100%|##########| 1/1 [00:03<00:00,  3.55s/it]


Config group #0: {'dp_noise_multiplier': None, 'dp': False, 'pretrain': True, 'config_ids': [0]}
Chunk_id: 0, # of syn dfs: 5, best_syndf: epoch_id-19.csv
Average truncation ratio: 0.0
Big syndf shape: (3935, 12)

Aggregated final dataset syndf
None (3935, 12)
best_syn_df filename: ./netshare_work_folder/post_processed_data/syn_df,dp_noise_multiplier-None,truncate-none,id-1.csv
Generated data is at ./netshare_work_folder/post_processed_data
Generated file list: ['./netshare_work_folder/post_processed_data/syn_df,dp_noise_multiplier-None,truncate-none,id-1.csv']
Best Gen trace: ./netshare_work_folder/post_processed_data/syn_df,dp_noise_multiplier-None,truncate-none,id-1.csv
Saved generated data to ../data/generated_netshare_caida_10k.csv
Ray is disabled
